# ⚓ Grounding Techniques

**Day 4 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- What grounding is and why it reduces hallucinations
- Prompt-based grounding: providing context the model must stay within
- Citation enforcement: requiring the model to reference its sources
- Fact-checking patterns: asking the model to verify its own claims
- Google Search grounding via the Gemini API

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. What Is Grounding?

**Grounding** means anchoring the model's responses to a specific, verifiable source of truth. Instead of relying on the model's parametric memory (training data), you supply the facts directly.

```
Without grounding:    [User question] → LLM → Answer (may hallucinate)

With grounding:       [User question] + [Verified context] → LLM → Answer (constrained to context)
```

Grounding strategies:
1. **Prompt-level grounding** — paste the source material directly into the prompt
2. **Citation enforcement** — require the model to cite every claim
3. **Self-verification** — ask the model to check its own output against the context
4. **Tool-based grounding** — use Google Search or a database to fetch real-time facts

---

## 2. Prompt-Level Grounding

The simplest form of grounding: **paste the source material** into the prompt and instruct the model to only use that material.

In [ ]:
# WITHOUT grounding — model may hallucinate details
response_ungrounded = client.models.generate_content(
    model=MODEL_ID,
    contents="What are the main features of the company's Enterprise plan?",
)
print("🔴 UNGROUNDED:")
print(response_ungrounded.text)
print()

In [ ]:
# WITH grounding — model is constrained to the provided document
product_docs = """
=== PRICING PLANS (Last updated: 2024-Q4) ===

STARTER PLAN ($0/month)
- Up to 3 users
- 5 GB storage
- Email support only
- No API access

PROFESSIONAL PLAN ($49/month per seat)
- Unlimited users
- 100 GB storage
- Priority email + chat support
- Full API access (10,000 calls/day)
- Custom integrations

ENTERPRISE PLAN (Custom pricing)
- Unlimited users and storage
- 24/7 phone + dedicated account manager
- Unlimited API access
- SSO and advanced security
- SLA: 99.99% uptime guarantee
- On-premise deployment option
"""

response_grounded = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Answer the question using ONLY the information in the document below.
If the answer is not in the document, say "This information is not available in the provided documentation."

<document>
{product_docs}
</document>

Question: What are the main features of the Enterprise plan?""",
)

print("🟢 GROUNDED:")
print(response_grounded.text)

---

## 3. Citation Enforcement

Require the model to **cite every claim** using inline references to the source material.

In [ ]:
research_excerpt = """
Source: "The Science of Sleep" (Johnson et al., 2023, Sleep Medicine Reviews)

[Paragraph 1] Adults require between 7 and 9 hours of sleep per night for optimal health. 
Consistently sleeping fewer than 6 hours is associated with a 48% increased risk of developing 
heart disease and a 15% increased risk of stroke.

[Paragraph 2] REM sleep, which occurs primarily in the second half of the night, is critical 
for memory consolidation and emotional regulation. Disruption of REM cycles is linked to 
increased anxiety and impaired decision-making.

[Paragraph 3] Exposure to blue light from screens within 2 hours of bedtime suppresses 
melatonin production by up to 22%, delaying sleep onset by an average of 40 minutes.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Summarize the key health implications of sleep patterns based on the document below.

REQUIREMENTS:
- Every factual claim must be followed by a citation in the format [Paragraph X]
- Do not add any information not present in the document
- If you cannot find support for a claim, do not make it

<document>
{research_excerpt}
</document>

Summarize with citations:""",
)

Markdown(response.text)

---

## 4. The "Only Use the Context" Instruction Pattern

A powerful system prompt pattern for grounded Q&A applications.

In [ ]:
GROUNDED_SYSTEM_PROMPT = """You are a helpful assistant that answers questions strictly based on the provided context.

Rules:
1. Only use information explicitly stated in the context
2. If the answer is not in the context, respond exactly: "I don't have enough information in the provided context to answer this."
3. Never infer, extrapolate, or use external knowledge
4. Quote directly when possible"""

faq_context = """
RETURN POLICY:
Items may be returned within 30 days of purchase with original receipt. 
Electronics have a 15-day return window. Items must be in original packaging.
Sale items are final sale and cannot be returned.

SHIPPING:
Standard shipping takes 5-7 business days and costs $5.99.
Express shipping (2-day) is available for $14.99.
Free shipping on orders over $75.
"""

questions = [
    "Can I return a laptop I bought 20 days ago?",
    "How much does shipping cost for a $50 order?",
    "Do you offer international shipping?",  # Not in context!
]

for question in questions:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[
            types.Content(role="user", parts=[types.Part(text=f"Context:\n{faq_context}\n\nQuestion: {question}")]),
        ],
        config=types.GenerateContentConfig(
            system_instruction=GROUNDED_SYSTEM_PROMPT,
        ),
    )
    print(f"Q: {question}")
    print(f"A: {response.text.strip()}")
    print("-" * 60)

---

## 5. Self-Verification Pattern (Chain-of-Verification)

Ask the model to **generate an answer, then verify it** against the source. This is a simple implementation of the Chain-of-Verification (CoVe) technique.

In [ ]:
company_facts = """
TechCorp was founded in 2015 by Elena Rodriguez and Marcus Webb.
The company is headquartered in Austin, Texas.
TechCorp has 1,200 employees as of 2024.
Annual revenue in 2023 was $340 million.
The company went public in 2021 on the NASDAQ under ticker TCH.
"""

question = "Who founded TechCorp and where is it located?"

# Step 1: Generate an answer
initial_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Based on this document, answer: {question}\n\n{company_facts}""",
)
print("📝 Initial Answer:")
print(initial_response.text)
print()

# Step 2: Verify the answer against the source
verification_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Verify this answer against the source document. 
For each claim in the answer, check whether it is:
- ✅ Supported by the document
- ❌ Contradicted by the document  
- ⚠️ Not mentioned in the document

<source_document>
{company_facts}
</source_document>

<answer_to_verify>
{initial_response.text}
</answer_to_verify>

Verification:""",
)

print("🔍 Verification:")
Markdown(verification_response.text)

---

## 6. Google Search Grounding

Gemini supports **real-time grounding via Google Search**. When enabled, the model fetches current information from the web before answering. This is especially useful for time-sensitive questions.

In [ ]:
# Google Search grounding — the model retrieves live web results
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the current version of Python and when was it released?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
    ),
)

print("Answer with Google Search grounding:")
print(response.text)
print()

# Check if grounding metadata was returned
if response.candidates[0].grounding_metadata:
    metadata = response.candidates[0].grounding_metadata
    print("\n📎 Grounding Sources:")
    if hasattr(metadata, 'grounding_chunks') and metadata.grounding_chunks:
        for chunk in metadata.grounding_chunks:
            if hasattr(chunk, 'web') and chunk.web:
                print(f"  - {chunk.web.title}: {chunk.web.uri}")
    else:
        print("  (Search was performed — grounding chunks not exposed in this API tier)")

---

## 🏋️ Exercise 1: Build a Grounded FAQ Bot

Create a grounded Q&A assistant that answers questions from a document you define.

In [ ]:
# Exercise 1: Grounded FAQ Bot
# TODO: Fill in your own document content below (any topic: a recipe, a policy, a product spec)

my_document = """
# TODO: Paste your own document here
# Example: a product manual, a company policy, a recipe, a news article, etc.
"""

# TODO: Write 3 questions — 2 that are answerable from the document, 1 that is not
test_questions = [
    # "Question 1 (answerable)",
    # "Question 2 (answerable)",
    # "Question 3 (NOT in the document)",
]

# TODO: Use the grounded system prompt pattern to build the FAQ bot
# Check that the model refuses to answer the out-of-scope question

for question in test_questions:
    pass  # Your code here

---

## 🏋️ Exercise 2: Citation Enforcement

Take the text below and ask the model to summarize it with mandatory inline citations.

In [ ]:
# Exercise 2: Citation enforcement

climate_excerpt = """
[Section A] Global average temperatures have risen by approximately 1.1°C since the pre-industrial period (1850-1900).
The last decade (2011-2020) was the warmest on record.

[Section B] Sea levels are rising at an accelerating rate. The global mean sea level rose by 20 cm 
between 1901 and 2018. The current rate of rise is 3.7 mm per year, more than double the rate 
observed in the 20th century.

[Section C] Extreme weather events are becoming more frequent and intense. Heatwaves that previously 
occurred once every 50 years now occur once every 10 years due to climate change.
"""

# TODO: Write a prompt that:
# 1. Asks for a 3-bullet-point summary of the key climate facts
# 2. Requires every bullet to include a citation to the section it came from
# 3. Prohibits adding any information not in the text

prompt = ""  # TODO: Write your prompt here

# response = client.models.generate_content(model=MODEL_ID, contents=prompt)
# Markdown(response.text)

---

## Key Takeaways

| Technique | When to Use | Limitation |
|-----------|-------------|------------|
| Prompt-level grounding | Short documents, single turn | Limited by context window |
| Citation enforcement | Reports, summaries, research | Model may still fabricate if not careful |
| Self-verification | High-stakes responses | Adds latency and cost |
| Google Search grounding | Real-time or recent information | Requires API access, internet connectivity |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Grounding with Google Search | Docs | [ai.google.dev/gemini-api/docs/grounding](https://ai.google.dev/gemini-api/docs/grounding) |
| Chain-of-Verification (CoVe) | Paper | [arxiv.org/abs/2309.11495](https://arxiv.org/abs/2309.11495) |
| RAG for Grounding (preview) | Coming in Day 7 | Retrieval-Augmented Generation |

---

**Next up:** [03_Generation_Control_Parameters.ipynb](./03_Generation_Control_Parameters.ipynb)